# Notebook 2 — Dot plot: chrF++ a nivel segmento vs juicio humano

Cruza el **chrF++ por segmento** (calculado en la Notebook 1, sección 1.2) con los
**juicios humanos** de una lingüista con experiencia en qom.

### Los datos de juicio humano

- **28 segmentos por dirección**, tomados del test estratificado de Base.
- Juicios **categóricos** (aproximadamente: correcto / incorrecto / dudoso).
- **No siguen el esquema MQM**: la tipificación MQM es una relectura posterior, así que
  **no hay severidades ni puntajes numéricos**.

> **No** se inventa un escalar de calidad ni se deriva uno a partir de las categorías.

### La pregunta de la figura

¿Los segmentos juzgados **incorrectos** se separan de los **correctos** en el eje de
chrF++, o se mezclan?

## Celda de configuración

**Completá `JUICIOS_CSV`.** Debe tener columnas `segment_id, direction, juicio_humano`.
`SEGMENT_CHRF_CSV` tiene que apuntar al **mismo archivo** que generó la Notebook 1.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 20260729
np.random.seed(RANDOM_STATE)

# ── Entradas (RELLENAR JUICIOS_CSV) ───────────────────────────────────────────
DATA_DIR = Path("RELLENAR/data")
JUICIOS_CSV = DATA_DIR / "RELLENAR_juicios_humanos.csv"   # segment_id, direction, juicio_humano

# Mismo path que en la Notebook 1 (chrF++ por segmento).
RESULTS_DIR = Path("resultados")
SEGMENT_CHRF_CSV = RESULTS_DIR / "chrf_por_segmento.csv"

# CSV de traducciones (para la tabla cualitativa de 2.4: fuente/referencia/hipótesis).
TRANSLATIONS_CSV = DATA_DIR / "RELLENAR_traducciones.csv"

# Salidas.
FIG_DIR = Path("../poster/figures")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Umbral por debajo del cual no se calculan correlaciones ni tests (n de la categoría).
N_MINIMO_TESTS = 10
N_BOOTSTRAP = 1000

# Sistema cuyo chrF++ por segmento se cruza con el juicio humano.
# Ajustá al sistema evaluado por la lingüista (p. ej. la variante estratificada de v2).
SISTEMA_EVALUADO = "RELLENAR-sistema-evaluado"

print("Config Notebook 2 cargada.")

In [ ]:
def _chequear_rellenar():
    faltan = [n for n, r in [("JUICIOS_CSV", JUICIOS_CSV)] if "RELLENAR" in str(r)]
    if faltan:
        raise ValueError("Completá estas rutas antes de seguir: " + ", ".join(faltan))
    if not Path(JUICIOS_CSV).exists():
        raise FileNotFoundError(f"No existe JUICIOS_CSV: {JUICIOS_CSV}")
    if not Path(SEGMENT_CHRF_CSV).exists():
        raise FileNotFoundError(
            f"No existe {SEGMENT_CHRF_CSV}. Corré primero la Notebook 1 (sección 1.2).")
    if "RELLENAR" in str(SISTEMA_EVALUADO):
        print("[aviso] SISTEMA_EVALUADO sin completar: se usará el único sistema presente "
              "en chrf_por_segmento.csv si hay uno solo; si hay varios, va a fallar claro.")

_chequear_rellenar()
print("Rutas OK.")

## 2.1 — Datos: carga, validación y cruce

Leemos el CSV de juicios y lo cruzamos por `segment_id` (y `direction`) con el chrF++ por
segmento. Validamos el formato y avisamos si hay **categorías inesperadas** o `segment_id`
que **no matcheen**. No completamos datos faltantes en silencio.

In [ ]:
# ── Normalización de direcciones (misma convención que la Notebook 1) ─────────
DIRECTION_ALIASES = {
    "qom2es": "qom2es", "qom-es": "qom2es", "qom_es": "qom2es", "qomes": "qom2es",
    "qom->es": "qom2es", "qom→es": "qom2es", "tob2spa": "qom2es",
    "es2qom": "es2qom", "es-qom": "es2qom", "es_qom": "es2qom", "esqom": "es2qom",
    "es->qom": "es2qom", "es→qom": "es2qom", "spa2tob": "es2qom",
}
def norm_direction(v):
    k = str(v).strip().lower().replace(" ", "")
    if k in DIRECTION_ALIASES:
        return DIRECTION_ALIASES[k]
    raise ValueError(f"Dirección no reconocida: {v!r}.")

# Categorías de juicio esperadas (normalizadas). Ajustá si tu esquema difiere.
CATEGORIAS_ESPERADAS = {"correcto", "incorrecto", "dudoso"}
CAT_ALIASES = {
    "correcto": "correcto", "correcta": "correcto", "ok": "correcto", "bien": "correcto",
    "si": "correcto", "sí": "correcto", "1": "correcto",
    "incorrecto": "incorrecto", "incorrecta": "incorrecto", "mal": "incorrecto",
    "no": "incorrecto", "0": "incorrecto",
    "dudoso": "dudoso", "dudosa": "dudoso", "duda": "dudoso", "?": "dudoso",
    "parcial": "dudoso",
}
def norm_categoria(v):
    k = str(v).strip().lower()
    return CAT_ALIASES.get(k, k)   # si no está en aliases, se deja tal cual y se avisa

# ── Carga de juicios ──────────────────────────────────────────────────────────
jr = pd.read_csv(JUICIOS_CSV)
print("Columnas juicios:", list(jr.columns))
lower = {c.lower().strip(): c for c in jr.columns}
def _col(*cands):
    for c in cands:
        if c in lower:
            return lower[c]
    raise ValueError(f"Falta alguna de estas columnas en {JUICIOS_CSV}: {cands}")

j = jr.rename(columns={
    _col("segment_id", "id", "seg_id"): "segment_id",
    _col("direction", "dir", "sentido", "direccion", "dirección"): "direction",
    _col("juicio_humano", "juicio", "categoria", "categoría", "label", "evaluacion",
         "evaluación"): "juicio_humano",
})[["segment_id", "direction", "juicio_humano"]].copy()
j["direction"] = j["direction"].map(norm_direction)
j["juicio_humano"] = j["juicio_humano"].map(norm_categoria)

# Aviso de categorías inesperadas.
inesperadas = set(j["juicio_humano"]) - CATEGORIAS_ESPERADAS
if inesperadas:
    print(f"[aviso] Categorías fuera de {CATEGORIAS_ESPERADAS}: {inesperadas}. "
          "Revisá CAT_ALIASES o el CSV. Se mantienen tal cual.")
print("\nConteo de juicios por dirección y categoría:")
print(j.groupby(["direction", "juicio_humano"]).size().to_string())

In [ ]:
# ── chrF++ por segmento y cruce ───────────────────────────────────────────────
chrf_seg = pd.read_csv(SEGMENT_CHRF_CSV)
chrf_seg["direction"] = chrf_seg["direction"].map(norm_direction)

# Elegir el sistema evaluado.
sistemas_disp = sorted(chrf_seg["system"].unique())
if "RELLENAR" in str(SISTEMA_EVALUADO):
    if len(sistemas_disp) == 1:
        sistema = sistemas_disp[0]
        print(f"SISTEMA_EVALUADO no fijado; uso el único presente: {sistema}")
    else:
        raise ValueError(
            f"Hay varios sistemas en chrf_por_segmento.csv ({sistemas_disp}). "
            "Fijá SISTEMA_EVALUADO en la config.")
else:
    sistema = SISTEMA_EVALUADO
    if sistema not in sistemas_disp:
        raise ValueError(f"'{sistema}' no está en chrf_por_segmento.csv. "
                         f"Disponibles: {sistemas_disp}")

chrf_sys = chrf_seg[chrf_seg["system"] == sistema][
    ["segment_id", "direction", "chrf_segment"]]

# Cruce por (segment_id, direction).
datos = j.merge(chrf_sys, on=["segment_id", "direction"], how="left")
sin_match = datos[datos["chrf_segment"].isna()]
if len(sin_match):
    print(f"[aviso] {len(sin_match)} juicios sin chrF++ (segment_id/direction que no "
          f"matchean con el sistema '{sistema}'):")
    print(sin_match[["segment_id", "direction", "juicio_humano"]].to_string(index=False))
datos = datos.dropna(subset=["chrf_segment"]).reset_index(drop=True)
print(f"\nSegmentos cruzados OK: {len(datos)} (sistema: {sistema}).")
datos.to_csv(RESULTS_DIR / "juicio_vs_chrf.csv", index=False)

## 2.2 — Figura principal (dot plot)

Un panel por dirección:

- eje **Y**: chrF++ del segmento;
- eje **X**: categoría de juicio humano, con **jitter** horizontal;
- **color y forma** de marcador según categoría (paleta apta para daltonismo);
- **mediana** de cada grupo con línea horizontal;
- **puntos individuales visibles**: no boxplot ni violín (con este n no corresponde).

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 18, "axes.titlesize": 22, "axes.labelsize": 20,
    "xtick.labelsize": 16, "ytick.labelsize": 16, "legend.fontsize": 16,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.constrained_layout.use": True,
})

# Okabe-Ito: color + forma redundantes por categoría (accesibilidad).
ESTILO_CAT = {
    "correcto":   {"color": "#009E73", "marker": "o"},   # verde, círculo
    "dudoso":     {"color": "#E69F00", "marker": "^"},   # naranja, triángulo
    "incorrecto": {"color": "#D55E00", "marker": "s"},   # bermellón, cuadrado
}
ORDEN_CAT = ["incorrecto", "dudoso", "correcto"]

def guardar(fig, nombre):
    fig.savefig(FIG_DIR / f"{nombre}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{nombre}.png", bbox_inches="tight", dpi=300)
    print(f"  guardada: {nombre}.pdf y .png (300 dpi)")

rng = np.random.default_rng(RANDOM_STATE)
direcciones = [d for d in ["qom2es", "es2qom"] if d in set(datos["direction"])]

fig, axes = plt.subplots(1, len(direcciones), figsize=(7 * len(direcciones), 7),
                         sharey=True, squeeze=False)
axes = axes[0]
for ax, direccion in zip(axes, direcciones):
    d = datos[datos["direction"] == direccion]
    cats = [c for c in ORDEN_CAT if c in set(d["juicio_humano"])]
    # categorías no previstas al final
    cats += [c for c in sorted(set(d["juicio_humano"])) if c not in cats]
    for x, cat in enumerate(cats):
        g = d[d["juicio_humano"] == cat]
        est = ESTILO_CAT.get(cat, {"color": "#0072B2", "marker": "D"})
        jitter = rng.uniform(-0.18, 0.18, size=len(g))
        ax.scatter(np.full(len(g), x) + jitter, g["chrf_segment"],
                   color=est["color"], marker=est["marker"], s=130,
                   edgecolor="white", linewidth=0.7, alpha=0.9, zorder=3)
        # Mediana del grupo.
        med = g["chrf_segment"].median()
        ax.plot([x - 0.28, x + 0.28], [med, med], color="#222222", lw=3, zorder=4)
        ax.text(x, -6, f"n={len(g)}", ha="center", va="top", fontsize=13, color="#555")
    ax.set_xticks(range(len(cats)))
    ax.set_xticklabels(cats)
    ax.set_title(direccion)
    ax.set_xlabel("juicio humano")
    ax.set_ylim(-8, 105)
axes[0].set_ylabel("chrF++ (segmento)")
fig.suptitle(f"chrF++ por segmento vs juicio humano — {sistema}")
guardar(fig, "figura_dotplot_juicio_chrf")
plt.show()

## 2.3 — Estadística descriptiva, con cautela

- **n por categoría y dirección.** Si alguna categoría tiene **menos de 10 ítems**, no se
  calculan coeficientes de correlación ni tests de hipótesis: se imprime una advertencia
  explícita.
- Sí se reportan **mediana, rango y rango intercuartílico** de chrF++ por categoría.
- Toda medida de asociación que se calcule va con su **IC bootstrap**.

In [ ]:
# ── Descriptivos por dirección x categoría ────────────────────────────────────
filas = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    for cat, g in d.groupby("juicio_humano"):
        x = g["chrf_segment"]
        filas.append({
            "direction": direccion, "categoria": cat, "n": len(g),
            "mediana": round(x.median(), 2),
            "min": round(x.min(), 2), "max": round(x.max(), 2),
            "q1": round(x.quantile(0.25), 2), "q3": round(x.quantile(0.75), 2),
            "iqr": round(x.quantile(0.75) - x.quantile(0.25), 2),
        })
descriptivos = pd.DataFrame(filas).sort_values(["direction", "categoria"])
print(descriptivos.to_string(index=False))
descriptivos.to_csv(RESULTS_DIR / "descriptivos_por_categoria.csv", index=False)

In [ ]:
# ── Asociación juicio-chrF++, sólo si el n lo permite ─────────────────────────
# Codificamos el juicio ordinalmente SÓLO para medir asociación monotónica (Spearman);
# esto no implica inventar un escalar de calidad, es un rango de las 3 categorías.
# Spearman = Pearson sobre rangos; lo implementamos con numpy para no sumar scipy.
ORDEN_ORDINAL = {"incorrecto": 0, "dudoso": 1, "correcto": 2}

def _rankdata(a):
    # Rangos con promedio en empates (equivalente a scipy.stats.rankdata 'average').
    a = np.asarray(a, dtype=float)
    orden = a.argsort(kind="mergesort")
    r = np.empty(len(a), dtype=float)
    sa = a[orden]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sa[j + 1] == sa[i]:
            j += 1
        r[orden[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return r

def spearman(x, y):
    rx, ry = _rankdata(x), _rankdata(y)
    if rx.std() == 0 or ry.std() == 0:
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])

def spearman_ic(x, y, n_boot, rng):
    rho = spearman(x, y)
    idx = np.arange(len(x))
    reps = []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        r = spearman(x[s], y[s])
        if not np.isnan(r):
            reps.append(r)
    lo, hi = np.percentile(reps, [2.5, 97.5]) if reps else (np.nan, np.nan)
    return rho, lo, hi

rng_b = np.random.default_rng(RANDOM_STATE + 7)
filas_assoc = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    conteos = d["juicio_humano"].value_counts()
    categorias_ok = all(conteos.get(c, 0) >= N_MINIMO_TESTS for c in ORDEN_ORDINAL)
    usables = d[d["juicio_humano"].isin(ORDEN_ORDINAL)]
    if not categorias_ok:
        chicas = {c: int(conteos.get(c, 0)) for c in ORDEN_ORDINAL}
        print(f"[{direccion}] n por categoría {chicas}: alguna < {N_MINIMO_TESTS}. "
              "NO se calculan correlaciones ni tests de hipótesis "
              "— el tamaño de muestra no lo permite.")
        continue
    x = usables["juicio_humano"].map(ORDEN_ORDINAL).to_numpy()
    y = usables["chrf_segment"].to_numpy()
    rho, lo, hi = spearman_ic(x, y, N_BOOTSTRAP, rng_b)
    filas_assoc.append({"direction": direccion, "spearman_rho": round(rho, 3),
                        "ic_low": round(lo, 3), "ic_high": round(hi, 3),
                        "n": len(usables)})

if filas_assoc:
    assoc = pd.DataFrame(filas_assoc)
    print(assoc.to_string(index=False))
    assoc.to_csv(RESULTS_DIR / "asociacion_spearman.csv", index=False)
else:
    print("No se calcularon medidas de asociación (ver avisos arriba).")

## 2.4 — Casos para inspección cualitativa

Segmentos **juzgados correctos con chrF++ más bajo** y **juzgados incorrectos con chrF++
más alto** (5 de cada uno por dirección), con fuente, referencia e hipótesis. Sirven como
ejemplos para el póster (dónde la métrica y el juicio humano se contradicen).

In [ ]:
# ── Traer fuente/referencia/hipótesis desde el CSV de traducciones ────────────
casos_out = []
if "RELLENAR" in str(TRANSLATIONS_CSV) or not Path(TRANSLATIONS_CSV).exists():
    print("[aviso] TRANSLATIONS_CSV no disponible: la tabla de casos saldrá sin "
          "fuente/referencia/hipótesis (sólo segment_id, juicio y chrF++).")
    tr_txt = None
else:
    tr_raw = pd.read_csv(TRANSLATIONS_CSV)
    low = {c.lower().strip(): c for c in tr_raw.columns}
    def _c(*cs):
        for c in cs:
            if c in low: return low[c]
        return None
    ren = {}
    for canon, cands in [("segment_id", ("segment_id", "id", "seg_id")),
                         ("direction", ("direction", "dir", "sentido", "direccion")),
                         ("system", ("system", "sistema", "model", "modelo")),
                         ("source", ("source", "src", "fuente", "entrada")),
                         ("reference", ("reference", "ref", "referencia")),
                         ("hypothesis", ("hypothesis", "hyp", "prediction", "pred"))]:
        col = _c(*cands)
        if col: ren[col] = canon
    tr_txt = tr_raw.rename(columns=ren)
    tr_txt["direction"] = tr_txt["direction"].map(norm_direction)
    tr_txt = tr_txt[tr_txt["system"] == sistema][
        ["segment_id", "direction", "source", "reference", "hypothesis"]]

def enriquecer(sub):
    if tr_txt is None:
        return sub
    return sub.merge(tr_txt, on=["segment_id", "direction"], how="left")

for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    correctos = d[d["juicio_humano"] == "correcto"].nsmallest(5, "chrf_segment")
    incorrectos = d[d["juicio_humano"] == "incorrecto"].nlargest(5, "chrf_segment")
    for etiqueta, sub in [("correcto_chrf_bajo", correctos),
                          ("incorrecto_chrf_alto", incorrectos)]:
        s = enriquecer(sub.copy())
        s.insert(0, "caso", etiqueta)
        casos_out.append(s)

if casos_out:
    casos = pd.concat(casos_out, ignore_index=True)
    cols = [c for c in ["caso", "direction", "segment_id", "juicio_humano",
                        "chrf_segment", "source", "reference", "hypothesis"]
            if c in casos.columns]
    casos = casos[cols]
    print(casos.to_string(index=False))
    casos.to_csv(RESULTS_DIR / "casos_cualitativos.csv", index=False)
    print(f"\nGuardado -> {RESULTS_DIR / 'casos_cualitativos.csv'}")

### Cierre

- La figura contesta de un vistazo si **incorrectos** y **correctos** se separan o se
  mezclan en el eje chrF++.
- Con estos tamaños de muestra, la estadística es **descriptiva**: correlaciones y tests
  sólo si cada categoría llega a `N_MINIMO_TESTS`, siempre con IC bootstrap.
- No se derivó ningún escalar de calidad a partir de los juicios categóricos.